In [26]:
import io

import pandas as pd
import requests

In [27]:
def read_ksan_obhistory(url="https://forecast.weather.gov/data/obhistory/KSAN.html"):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; research script)"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()

    tables = pd.read_html(io.StringIO(resp.text))
    # This page has one main data table; find the one with the most rows
    df = max(tables, key=len)

    return df

df = read_ksan_obhistory()
print(df.head())
print(df.columns)

  Date Time (pdt) Wind (mph) Vis. (mi.)   Weather Sky Cond. Temperature (ºF)  \
  Date Time (pdt) Wind (mph) Vis. (mi.)   Weather Sky Cond.              Air   
  Date Time (pdt) Wind (mph) Vis. (mi.)   Weather Sky Cond.              Air   
0   30      05:51       Calm      10.00  Overcast    OVC013               68   
1   30      04:51       Calm      10.00  Overcast    OVC014               68   
2   30      03:51      NW  5      10.00  Overcast    OVC014             69.1   
3   30      02:51      NE  6      10.00  Overcast    OVC014             69.1   
4   30      01:51       S  5      10.00  Overcast    OVC013             69.1   

                    Relative Humidity Wind Chill (°F) Heat Index (°F)  \
   Dwpt 6 hour      Relative Humidity Wind Chill (°F) Heat Index (°F)   
   Dwpt   Max. Min. Relative Humidity Wind Chill (°F) Heat Index (°F)   
0    63    NaN  NaN               84%             NaN             NaN   
1  62.1     70   68               81%             NaN             N

In [28]:
df.columns = [
       "Date", "Time", "Wind", "Vis", "Weather", "Sky",
       "TempAir", "TempDwpt", "Temp6hrMax", "Temp6hrMin",
       "RH", "WindChill", "HeatIndex", "Altimeter", "SeaLevelPressure",
       "Precip1hr", "Precip3hr", "Precip6hr"
   ]

In [29]:
df = df[df["Date"] != "Date"].reset_index(drop=True)

In [30]:
df.head(2)

,Date,Time,Wind,Vis,Weather,Sky,TempAir,TempDwpt,Temp6hrMax,Temp6hrMin,RH,WindChill,HeatIndex,Altimeter,SeaLevelPressure,Precip1hr,Precip3hr,Precip6hr
0,30,05:51,Calm,10.00,Overcast,OVC013,68,63,NaN,NaN,84%,NaN,NaN,29.91,1012.7,NaN,NaN,NaN
1,30,04:51,Calm,10.00,Overcast,OVC014,68,62.1,70,68,81%,NaN,NaN,29.9,1012.4,NaN,NaN,NaN


In [31]:

from datetime import datetime

def add_full_date(df, as_of=None, date_col="Date", time_col="Time (pdt)"):
    """
    Reconstructs full dates for the obhistory table, which only shows day-of-month.
    Assumes df is ordered newest-first (as scraped from the page).

    as_of: datetime representing "today" at scrape time. Defaults to now.
    """
    df = df.copy()
    if as_of is None:
        as_of = datetime.now()

    current_year = as_of.year
    current_month = as_of.month

    full_dates = []
    prev_day = None

    for day in df[date_col].astype(int):
        if prev_day is not None and day > prev_day:
            # Day number went UP while moving backward in time -> month rolled back
            current_month -= 1
            if current_month == 0:
                current_month = 12
                current_year -= 1
        full_dates.append(datetime(current_year, current_month, day))
        prev_day = day

    df["FullDate"] = full_dates

    # Optional: combine with the time column into a full timestamp
    if time_col in df.columns:
        df["Timestamp"] = pd.to_datetime(
            df["FullDate"].dt.strftime("%Y-%m-%d") + " " + df[time_col],
            errors="coerce"
        )

    return df

In [42]:
from zoneinfo import ZoneInfo

In [43]:
datetime.now(tz=ZoneInfo("America/Los_Angeles"))

datetime.datetime(2026, 7, 30, 7, 3, 38, 471404, tzinfo=zoneinfo.ZoneInfo(key='America/Los_Angeles'))

In [44]:
datetime.now(tz=ZoneInfo("Europe/London"))

datetime.datetime(2026, 7, 30, 15, 4, 4, 755264, tzinfo=zoneinfo.ZoneInfo(key='Europe/London'))

In [32]:
df = add_full_date(df)

In [33]:
df.head(2)

,Date,Time,Wind,Vis,Weather,Sky,TempAir,TempDwpt,Temp6hrMax,Temp6hrMin,RH,WindChill,HeatIndex,Altimeter,SeaLevelPressure,Precip1hr,Precip3hr,Precip6hr,FullDate
0,30,05:51,Calm,10.00,Overcast,OVC013,68,63,NaN,NaN,84%,NaN,NaN,29.91,1012.7,NaN,NaN,NaN,2026-07-30
1,30,04:51,Calm,10.00,Overcast,OVC014,68,62.1,70,68,81%,NaN,NaN,29.9,1012.4,NaN,NaN,NaN,2026-07-30


In [34]:
df['Precip1hr'] = df.Precip1hr.fillna(0)

In [35]:
daily_weather = df.groupby('FullDate').TempAir.agg(['min', 'max']).reset_index()

In [36]:
precip_df = df.groupby('FullDate').Precip1hr.agg(['sum']).reset_index()

In [37]:
precip_df.rename(columns={'sum':'PRCP'}, inplace=True)

In [38]:
daily_weather = daily_weather.merge(precip_df,
                    how='left',
                    on='FullDate')

In [39]:
daily_weather

,FullDate,min,max,PRCP
0,2026-07-27,70,78.1,0
1,2026-07-28,69.1,77,0
2,2026-07-29,69.1,77,0
3,2026-07-30,68,69.1,0


In [ ]:
def